# Q27 · vLLM 0.32.0 · 4×L4 · Consolidated Deep-Audited v3 (1500+ target)

**Built from both uploaded notebooks, retaining every original vLLM multi-arm experiment.**

- `PLAN` default: GPU/OFFLINE INSTALL OFF, prints arms and preflight.
- `TEST`: same CPU checks, no GPU.
- `RUN`: offline installation + sequential independent arms and strict GPU-release gate.
- Suggested initial arms: `V0_FP8_ORACLE_EAGER`, `V0_FP8_BASE`, `V1_MTP_LOCAL_ARGMAX`.
- Real numbers only. `WAVE / accepted tokens ÷ entire HTTP wall`, and `STEADY / engine generation-counter post-prefill` are different.
- **Not production validated**: mock CPU tests do not prove CUDA correctness, KV fit, MTP-state behavior or 1500 tok/s.
- Research-only patch arms require explicit executable/source SHA-256 plus activation evidence; missing artifacts become SKIPPED, not PASS.
- Numerical oracle is raw generated token IDs from the same weights, same session, same tokenizer; high-margin divergence blocks performance promotion.
- 90k/130k profile requires actual `MODEL_LEN`, GPU fit and enough time; skipped points never count as a full-profile winner.
- Each arm runs a new server and stops on any GPU-release ambiguity.


In [ ]:
import os, sys, json, time, re, csv, hashlib, subprocess, signal, shutil, socket, statistics
import urllib.request, urllib.error, concurrent.futures, threading, pathlib, math, contextlib, uuid
from collections import defaultdict
from datetime import datetime, timezone
P = pathlib.Path

# --------------------------- USER SETTINGS ---------------------------
MODE = os.environ.get("Q27_SPEEDLAB_MODE", "PLAN").upper()  # PLAN | TEST | RUN
SELECTED_ARMS = [
    "V0_FP8_ORACLE_EAGER", "V0_FP8_BASE", "V1_MTP_LOCAL_ARGMAX",
]  # See all names in the arm inventory below.
PROFILE = os.environ.get("Q27_BENCH_PROFILE", "SMOKE").upper()  # SMOKE | STANDARD | XL | TRACE
MAX_LAB_MINUTES = float(os.environ.get("Q27_LAB_MINUTES", "170"))
MAX_ARM_MINUTES = float(os.environ.get("Q27_ARM_MINUTES", "22"))
MODEL_LEN = int(os.environ.get("Q27_MODEL_LEN", "106496"))  # set >= 135168 for 130k
FP8_MODEL = P(os.environ.get("Q27_FP8_MODEL", "/kaggle/input/models/foysalemonshanto/qwen3-8-27b-fp8-repacked-v1/pytorch/hf-fp8/1"))
INT4_MODEL = P(os.environ.get("Q27_INT4_MODEL", "/kaggle/input/models/foysalemonshanto/qwen3-8-27b-int4-redhat/pytorch/w4a16-g128/1"))
MERLIN_MODEL = P(os.environ.get("Q27_MERLIN_MODEL", "/kaggle/input/ATTACH-MERLIN-TP1-CHECKPOINT"))
WHEELHOUSE = P(os.environ.get("Q27_VLLM_WHEELHOUSE", "/kaggle/input/datasets/foysalemonshanto/vllm-0-32-0-cu13-wheelhouse"))
VENV = P(os.environ.get("Q27_VLLM_VENV", "/tmp/vllm-venv"))
PATCH_ROOT = P(os.environ.get("Q27_PATCH_ROOT", "/kaggle/input/q27-vllm-patched-artifacts"))
EAGLE_SPEC_FILE = P(os.environ.get("Q27_EAGLE4_JSON", "/kaggle/input/ATTACH-EAGLE4-SPEC.json"))
DFLASH_SPEC_FILE = P(os.environ.get("Q27_DFLASH_JSON", "/kaggle/input/ATTACH-DFLASH-SPEC.json"))
TRACE_JSONL = P(os.environ.get("Q27_TRACE_JSONL", "/kaggle/input/ATTACH-AGENT-TRACE.jsonl"))
NCCL_TEST_BIN = P(os.environ.get("Q27_ALLREDUCE_PERF", "/kaggle/input/nccl-tests/all_reduce_perf"))
OUTPUT_ROOT = P("/kaggle/working/q27_speedlab" if P("/kaggle/working").exists() else "/mnt/data/q27_speedlab")
SESSION_ID = uuid.uuid4().hex[:12]
OUTPUT = OUTPUT_ROOT / ("session_" + SESSION_ID)
OUTPUT.mkdir(parents=True, exist_ok=True)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
(OUTPUT_ROOT / "latest_session.txt").write_text(str(OUTPUT))
BASE_PORT = int(os.environ.get("Q27_PORT", "18110"))
STARTUP_LIMIT_SEC = int(os.environ.get("Q27_STARTUP_SEC", "590"))
HTTP_TIMEOUT_SEC = int(os.environ.get("Q27_HTTP_TIMEOUT_SEC", "210"))
ENABLE_STEADY_WINDOWS = os.environ.get("Q27_ENABLE_STEADY", "1") == "1"
STEADY_WINDOW_SECONDS = float(os.environ.get("Q27_STEADY_SECONDS", "18"))
STEADY_ADMISSION_SECONDS = float(os.environ.get("Q27_STEADY_ADMIT_SECONDS", "145"))
ORACLE_BANDS = [int(x) for x in os.environ.get("Q27_ORACLE_BANDS", "8192,40000").split(',') if x.strip()]
MIN_SYNTHETIC_COMPLETION_FRACTION = float(os.environ.get("Q27_MIN_COMPLETION_FRACTION", "0.80"))
STREAM_BENCHMARK = os.environ.get("Q27_STREAM_BENCHMARK", "1") == "1"
SEED=20261005
DISALLOW_UNBOUNDED_NETWORK=True
print('MODE',MODE,'PROFILE',PROFILE,'SELECTED_ARMS',SELECTED_ARMS)
print('FP8',FP8_MODEL,'INT4',INT4_MODEL,'wheelhouse',WHEELHOUSE,'output',OUTPUT)

## 1. Arm registry — আগে একবারে মাত্র একটি independent change

নিচের arms-এর তিনটি class:

- **NATIVE:** pinned vLLM0.32 wheel/flags দিয়ে পরীক্ষা করা যায়। CLI এবং runtime support অবশ্যই probe হবে।
- **PATCH:** `PATCH_ROOT/<patch_id>/manifest.json` + `venv/bin/vllm` ছাড়া চলবে না; manifest-এ exact upstream PR/source SHA উল্লেখ থাকতে হবে। Metadata alone correctness proof নয়।
- **EXTERNAL MODEL:** Merlin/TP1 or EAGLE-4 dataset না থাকলে `SKIPPED_REQUIRED_ARTIFACT`।

Research tracks user-requested: **#1 TP1×4/PP4; #2 host-staged AR; #3 graph verification; #4 native FP8/low-M; #5 NVFP4 KV; #6 adaptive/EAGLE speculation**, plus profiler/local-argmax/GDN/NCCL/MTP/prefix/int4. `PROFILE="XL"` কঠিন memory-fit test, `TRACE` real agent JSONL।

In [ ]:
# Registry items are plain serializable dictionaries; experiment baseline is never mutated.
ARMS={}
def add(name, track, weight="fp8", req="native", **opts):
    assert name not in ARMS
    ARMS[name] = {"name":name,"track":track,"weight":weight,"requirement":req,**opts}

# Frozen target-only oracles. Independent from baseline, eager, no speculation, no prefix-cache.
add("V0_FP8_ORACLE_EAGER","oracle",seqs=4,spec=0,enforce_eager=True,no_prefix=True)
add("V0_INT4_ORACLE_EAGER","oracle",weight="int4",seqs=4,spec=0,enforce_eager=True,no_prefix=True)
# Controls
add("V0_FP8_BASE","control", seqs=16, spec=3)
add("V0_FP8_TARGET_ONLY","control", seqs=16, spec=0)
add("V0_INT4_BASE","control", weight="int4", seqs=20, spec=3)
add("V0_INT4_TARGET_ONLY","control", weight="int4", seqs=16, spec=0)
add("V0_PROFILE_TORCH","profiling",seqs=16,spec=3,torch_profiler=True)
add("V0_PROFILE_NSYS","profiling", seqs=16, spec=3, nsys=True)

# First GPU campaign: TP vocab reduction / graphs / state dtype / NCCL / MTP
add("V1_MTP_LOCAL_ARGMAX","local_argmax",seqs=16,spec=3, spec_extra={"use_local_argmax_reduction":True})
add("V2_TRITON_FULL_AND_PIECEWISE","graphs",seqs=16,spec=3, backend="TRITON_ATTN", graph="FULL_AND_PIECEWISE")
add("V2_TRITON_FULL_DECODE_ONLY","graphs",seqs=16,spec=3, backend="TRITON_ATTN", graph="FULL_DECODE_ONLY")
add("V2_FLASHINFER_FULL_PR60532","graphs",req="patch_60532",seqs=16,spec=3, graph="FULL_AND_PIECEWISE")
add("V3_GDN_FP16","ssm_state",seqs=16,spec=3,ssm="float16")
add("V3_GDN_BF16","ssm_state",seqs=16,spec=3,ssm="bfloat16")
for cta in (1,2,4,8):
    add(f"V4_NCCL_CTA_{cta}","nccl",seqs=16,spec=3,env={"NCCL_MIN_CTAS":str(cta),"NCCL_MAX_CTAS":str(cta)})
for alg in ("Ring","Tree"):
    add(f"V4_NCCL_{alg.upper()}","nccl",seqs=16,spec=3,env={"NCCL_ALGO":alg})
add("V4_NCCL_NTHREADS512","nccl",seqs=16,spec=3,env={"NCCL_NTHREADS":"512"})
add("V4_NCCL_TESTS","nccl_micro",req="nccl_bin")
add("V5_MTP1","spec_depth",seqs=16,spec=1)
add("V5_MTP3","spec_depth",seqs=16,spec=3)
add("V5_TARGET_ONLY","spec_depth",seqs=16,spec=0)
for unit in (16,64):
    add(f"V6_PREFIX{unit}","hybrid_prefix",seqs=16,spec=3,extra=["--prefix-match-unit",str(unit)])
    add(f"V6_SHARED{unit}","hybrid_prefix",seqs=16,spec=3,extra=["--prefix-match-unit",str(unit),"--enable-mamba-shared-prefix-checkpoint"])
add("V6_KEEP_TRAILING_BLOCK","hybrid_prefix",seqs=16,spec=3,spec_extra={"disable_eagle_block_drop":True})
add("V6_PREFIX16_AND_KEEP_BLOCK","hybrid_prefix",seqs=16,spec=3,
    extra=["--prefix-match-unit","16","--enable-mamba-shared-prefix-checkpoint"],
    spec_extra={"disable_eagle_block_drop":True})
add("V7_INT4_S20","weight_quant",weight="int4",seqs=20,spec=3)

# High-upside source backports (require external independently built offline artifacts)
add("V8_SM89_CUTLASS_FP8_PR61224","lowm_fp8",req="patch_61224",seqs=16,spec=3)
add("V8_MERLIN_LOWM_FORK","lowm_fp8",req="patch_merlin",seqs=16,spec=3)
add("V9_NVFP4_KV_TARGET_ONLY","nvfp4_kv",req="patch_46963",seqs=12,spec=0,kv="nvfp4")
add("V9_NVFP4_KV_MTP3","nvfp4_kv",req="patch_46963",seqs=12,spec=3,kv="nvfp4")
add("V10_EAGLE4","eagle4",req="eagle_spec",seqs=12,spec="external")
add("V10_DFLASH_EXTERNAL","eagle4",req="draft_spec",seqs=12,spec="external")

# Architecture: one full model per GPU vs pipeline 4 stage. TP1 model may need host offload / special quant.
add("V11_TP1_SINGLE","parallelism",req="patch_merlin_tp1",weight="merlin",tp=1,visible="0",seqs=2,spec=0)
add("V11_TP1X2_REPLICAS","parallelism",req="patch_merlin_tp1",weight="merlin",tp=1,replicas=2,seqs=2,spec=0)
add("V11_TP1X4_REPLICAS","parallelism",req="patch_merlin_tp1",weight="merlin",tp=1,replicas=4,seqs=2,spec=0)
add("V11_PP4_TARGET_ONLY","parallelism",tp=1,pp=4,seqs=8,spec=0)
add("V11_PP4_MTP3","parallelism",tp=1,pp=4,seqs=8,spec=3)
add("V12_HOST_STAGED_AR","custom_ar",req="patch_host_ar",seqs=16,spec=3,custom_ar=True)
add("V12_HOST_STAGED_INT8_AR","custom_ar",req="patch_host_ar_int8",seqs=16,spec=3,custom_ar=True)
add("V13_PREFILL2048","scheduler",seqs=16,spec=3,batched_tokens=2048)
add("V13_PREFILL8192","scheduler",seqs=16,spec=3,batched_tokens=8192)
add("V13_DBO","scheduler",seqs=16,spec=3,extra=["--enable-dbo"])
for n in (12,16,20,24):
    add(f"V14_STREAMS_{n}","concurrency",seqs=n,spec=3)

assert len(ARMS)==len(set(ARMS))
for name in ARMS:
    a=ARMS[name]
    print(f"{name:<38} {a['track']:<17} {a['requirement']}")

## 2. Offline installer, artifact manifests এবং fail-closed preflight

Baseline wheelhouse থেকে `install_wheelhouse.py` ব্যবহার করা হয়—E22-এর tested installation pathway অনুযায়ী। **Internet থেকে pip install বা model download হয় না।** Patched arm-এ আলাদা binary/venv দরকার। Manifest example: `PATCH_ROOT/60532/manifest.json` containing `{"vllm_version":"0.32.0","patches":["60532"],"vllm_executable":"/kaggle/input/.../venv/bin/vllm", "source_sha256":"..."}`. Wheel import path, patch version/hash এবং runtime graph evidence আরও পরীক্ষা করতে হবে।

**`nvfp4` dtype accepted ≠ actual SM89 kernels validated**; graph/compile unsupported হলে arm FAIL/INCONCLUSIVE। TP1×4 check শুধু replication hardware feasibility, 90k density/accuracy অনুমান নয়।

In [ ]:
def parse_version(s):
    m = re.search(r'\b0\.32\.0\b',s); return bool(m)
def choose_model(a):
    return {"fp8":FP8_MODEL,"int4":INT4_MODEL,"merlin":MERLIN_MODEL}[a.get("weight","fp8")]
def artifact_for(a):
    req=a["requirement"]
    if req=="native": return None
    if req in ("merlin_model",): return {"model":str(MERLIN_MODEL)}  # legacy explicit model-only validation
    if req in ("eagle_spec", "draft_spec"):
        return {"spec":str(EAGLE_SPEC_FILE if req=="eagle_spec" else DFLASH_SPEC_FILE)}
    if req=="nccl_bin": return {"bin":str(NCCL_TEST_BIN)}
    if req.startswith("patch_"):
        key=req[6:]
        return {"manifest":str(PATCH_ROOT/key/"manifest.json")}
    return {"unknown_requirement":req}
def model_shards_ok(root):
    root=P(root)
    if not (root/'config.json').is_file():return (False,'config.json missing')
    import glob
    indexes=list(root.glob('*.safetensors.index.json'))+list(root.glob('*.bin.index.json'))
    if indexes:
        try:
            shards=set()
            for p in indexes:
                d=json.loads(p.read_text()); shards.update(d['weight_map'].values())
            if not shards:return (False,'empty model weight index')
            if any(P(x).is_absolute() or '..' in P(x).parts for x in shards):
                return (False,'unsafe shard path in index')
            missing=[x for x in shards if not (root/x).is_file()]
            if missing:return (False,'missing weight shards: '+repr(missing[:4]))
        except Exception as e:return (False,'invalid weight index: '+repr(e))
        return (True,None)
    files=list(root.glob('*.safetensors'))+list(root.glob('pytorch_model*.bin'))
    if not files:return (False,'No safetensors or pytorch_model.bin shards')
    return (True,None)

def validate_artifact(a):
    req=a["requirement"]
    if req=="native": return (True,None)
    if req=="merlin_model":
        return model_shards_ok(MERLIN_MODEL)
    if req=="nccl_bin": return (NCCL_TEST_BIN.is_file() and os.access(NCCL_TEST_BIN,os.X_OK),"nccl-tests binary missing")
    if req in ("eagle_spec","draft_spec"):
        path=EAGLE_SPEC_FILE if req=="eagle_spec" else DFLASH_SPEC_FILE
        if not path.is_file(): return False,"external spec JSON missing: "+str(path)
        try:
            d=json.loads(path.read_text())
            if not isinstance(d,dict) or not isinstance(d.get("method"),str) or d["method"] in ("mtp", ""):
                return False,"external draft spec must specify a non-MTP method"
            if req=="eagle_spec" and "eagle" not in d["method"].lower():
                return False,"EAGLE4 arm requires an EAGLE-family method"
            if req=="draft_spec" and "dflash" not in d["method"].lower():
                return False,"DFlash arm requires a DFlash method"
            mp=d.get('model') or d.get('draft_model')
            if not isinstance(mp,str):
                return False,"external drafter model path missing"
            ok,why=model_shards_ok(P(mp))
            if not ok:return False,"external drafter checkpoint incomplete: "+str(why)
        except Exception as e: return False,repr(e)
        return True,None
    if req.startswith("patch_"):
        k=req[6:]; mf=PATCH_ROOT/k/"manifest.json"
        if not mf.is_file(): return False,"patch manifest missing: "+str(mf)
        try:
            d=json.loads(mf.read_text())
            exe=P(d.get("vllm_executable",""))
            if d.get("vllm_version")!="0.32.0" or k not in d.get("patches",[]):
                return False,"manifest version or patch ID mismatch"
            if not exe.is_file() or not os.access(exe,os.X_OK):
                return False,"patched executable missing"
            sh=d.get('vllm_executable_sha256')
            if not isinstance(sh,str) or not re.fullmatch(r'[a-fA-F0-9]{64}',sh):
                return False,'patched executable SHA256 mandatory'
            if hashlib.sha256(exe.read_bytes()).hexdigest().lower()!=sh.lower():
                return False,'PATCH_EXECUTABLE_SHA256_MISMATCH' 
            source=P(d.get('source_file',''))
            if not re.fullmatch(r"[a-fA-F0-9]{64}", d.get("source_sha256","")) or not source.is_file():
                return False,"patch source_file and SHA256 are required"
            if hashlib.sha256(source.read_bytes()).hexdigest().lower()!=d['source_sha256'].lower():
                return False,"PATCH_SOURCE_SHA256_MISMATCH"
            if not isinstance(d.get('activation_regex'),str) or not d['activation_regex'].strip():
                return False,"patch activation_regex must identify the actual loaded kernel/graph/collective"
        except Exception as e: return False,repr(e)
        return True,None
    return False,"unrecognized requirement"

def discover_installer():
    if not WHEELHOUSE.exists(): return None
    matches=list(WHEELHOUSE.rglob("install_wheelhouse.py"))
    return sorted(matches)[0] if matches else None

def ensure_baseline_venv():
    exe=VENV/"bin/vllm"
    if not exe.is_file():
        inst=discover_installer()
        if not inst: raise FileNotFoundError("Offline install_wheelhouse.py not found under "+str(WHEELHOUSE))
        r=subprocess.run([sys.executable,str(inst),str(VENV)],text=True,capture_output=True,timeout=2400,
                         env={**os.environ,"PIP_NO_INDEX":"1","HF_HUB_OFFLINE":"1","TRANSFORMERS_OFFLINE":"1"})
        (OUTPUT/"install.log").write_text((r.stdout+"\n"+r.stderr)[-100000:],encoding="utf-8")
        kv=dict(line.split('=',1) for line in r.stdout.splitlines() if line.startswith(('CUDA_HOME=','VENV_PYTHON=')))
        if kv.get('CUDA_HOME'):
            atomic_json(OUTPUT/'install_cuda_env.json',{'CUDA_HOME':kv['CUDA_HOME']})
        if r.returncode: raise RuntimeError(f"offline install failed rc={r.returncode}; see install.log")
    ver=subprocess.run([str(VENV/"bin/python"),"-c","import vllm;print(vllm.__version__)"],
                      text=True,capture_output=True,timeout=120)
    if ver.returncode or not parse_version(ver.stdout):
        raise RuntimeError("Installed vllm != 0.32.0: "+(ver.stdout+ver.stderr)[-1200:])
    return exe

def cuda_env(exe, a):
    env={k:v for k,v in os.environ.items() if k not in ("PYTHONPATH","CUDA_PATH")}
    env.update(HF_HUB_OFFLINE="1",TRANSFORMERS_OFFLINE="1",PIP_NO_INDEX="1",
               VLLM_NO_USAGE_STATS="1",VLLM_ALLOW_LONG_MAX_MODEL_LEN="1",PYTHONUNBUFFERED="1",
               TOKENIZERS_PARALLELISM="false",VLLM_CACHE_ROOT=str(OUTPUT/"vllm_cache"))
    shim=P("/tmp/q27_libcuda"); shim.mkdir(exist_ok=True)
    for root in ("/usr/local/nvidia/lib64","/usr/local/cuda/compat","/usr/lib/x86_64-linux-gnu","/usr/lib64"):
        src=P(root)/"libcuda.so.1"
        if src.is_file() and not (shim/"libcuda.so").exists():
            (shim/"libcuda.so").symlink_to(src)
    env["LIBRARY_PATH"]=str(shim)+":"+env.get("LIBRARY_PATH","")
    env["PATH"]=str(P(exe).parent)+":"+env.get("PATH","")
    kvenv=OUTPUT/'install_cuda_env.json'
    if kvenv.is_file():
        cfg=json.loads(kvenv.read_text()); env['CUDA_HOME']=cfg['CUDA_HOME']
        env['PATH']=cfg['CUDA_HOME']+'/bin:'+env['PATH']
    elif os.environ.get('CUDA_HOME'):env['CUDA_HOME']=os.environ['CUDA_HOME']
    if a.get("visible") is not None: env["CUDA_VISIBLE_DEVICES"]=str(a["visible"])
    env.update(a.get("env",{}))
    if a.get("requirement","").startswith("patch_"):
        d=json.loads(P(artifact_for(a)["manifest"]).read_text())
        env.update({str(k):str(v) for k,v in d.get("env",{}).items()})
    return env

if MODE not in ("PLAN","TEST","RUN"): raise ValueError("MODE must be PLAN, TEST or RUN")
preflight={}
for an in SELECTED_ARMS:
    if an not in ARMS: raise KeyError("Unknown selected arm: "+an)
    a=ARMS[an]; ok,reason=validate_artifact(a)
    # Candidate model files missing => skip, never marked as success.
    if a['track']!='nccl_micro':
        ready,why=model_shards_ok(choose_model(a))
        if not ready:ok=False; reason="checkpoint incomplete at "+str(choose_model(a))+": "+str(why)
    preflight[an]={'artifact_ok':ok,'detail':reason,'required':artifact_for(a)}
print(json.dumps(preflight,indent=2,ensure_ascii=False))

## 3. Build exact CLI, process isolation এবং server lifecycle

CLI flags version-compatible কি না `vllm serve --help` থেকে পরীক্ষা করা হয়। **Requested FULL graphs automatically prove FULL capture নয়**; actual runtime log/trace record থাকবে। Patch-backed server executable `manifest`-specified; heuristic no-op flag changes allowed নয়।

In [ ]:
def spec_json(a):
    depth=a.get('spec',3)
    if depth=="external":
        p=DFLASH_SPEC_FILE if a.get('requirement')=='draft_spec' else EAGLE_SPEC_FILE
        return json.dumps(json.loads(p.read_text()),separators=(',',':'))
    if int(depth)<=0: return None
    return json.dumps({"method":"mtp","num_speculative_tokens":int(depth),
                       **a.get('spec_extra',{})},separators=(',',':'))

def cli_for(a, port, visible=None):
    tp=int(a.get('tp',4)); pp=int(a.get('pp',1)); seqs=int(a.get('seqs',16)); max_len=int(a.get('max_len',MODEL_LEN))
    if a.get('replicas',1)>1: seqs=min(seqs,4)
    flags=[str(choose_model(a)),"--served-model-name","q27","--host","127.0.0.1","--port",str(port),
           "--tensor-parallel-size",str(tp),"--pipeline-parallel-size",str(pp),
           "--max-num-seqs",str(seqs),"--max-num-batched-tokens",str(a.get("batched_tokens",4096)),
           "--max-model-len",str(max_len),"--gpu-memory-utilization","0.90", "--kv-cache-dtype",a.get('kv','fp8_e4m3'),
           "--attention-backend",a.get('backend','FLASHINFER'),"--seed",str(SEED),"--language-model-only",
           "--generation-config","vllm",
           "--trust-remote-code","--dtype","auto","--load-format","safetensors",
           "--distributed-executor-backend","mp","--enable-prompt-tokens-details",
           "--enable-auto-tool-choice","--tool-call-parser","qwen3_coder",
           "--reasoning-parser","qwen3","--default-chat-template-kwargs",
           json.dumps({"preserve_thinking":True}),"--disable-uvicorn-access-log"]
    if a.get("no_prefix"): flags.append("--no-enable-prefix-caching")
    else: flags.append("--enable-prefix-caching")
    if a.get("enforce_eager"): flags.append("--enforce-eager")
    if a.get("torch_profiler"):
        flags.extend(['--profiler-config',json.dumps({'profiler':'torch','torch_profiler_dir':str(OUTPUT/(a['name']+'_torch_traces'))})])
    if not a.get('custom_ar'): flags.append('--disable-custom-all-reduce')
    spec=spec_json(a)
    if spec: flags += ['--speculative-config',spec]
    if a.get('graph'): flags += ['--compilation-config',json.dumps({'cudagraph_mode':a['graph']})]
    if a.get('ssm'): flags += ['--mamba-ssm-cache-dtype',a['ssm']]
    flags.extend(a.get('extra',[]))
    if a.get('requirement','').startswith('patch_'):
        mf=P(artifact_for(a)['manifest'])
        if mf.is_file():
            d=json.loads(mf.read_text())
            flags.extend(d.get('extra_args',[]))
        # Missing manifest is rejected by validate_artifact before any RUN launch.
    return flags

def exe_for(a, baseline_exe):
    if a['requirement'].startswith('patch_'):
        return P(json.loads(P(artifact_for(a)['manifest']).read_text())['vllm_executable'])
    return P(baseline_exe)

def help_for(exe,env):
    p=subprocess.run([str(exe),'serve','--help'],capture_output=True,text=True,env=env,timeout=130)
    if p.returncode: raise RuntimeError('vllm help failed '+(p.stdout+p.stderr)[-1500:])
    return p.stdout+p.stderr

def validate_flags(flags, help_text):
    # --host etc present in v0.32. Only check the optional candidate flags.
    special={'--compilation-config','--mamba-ssm-cache-dtype','--prefix-match-unit',
             '--enable-mamba-shared-prefix-checkpoint','--enable-dbo','--pipeline-parallel-size',
             '--kv-cache-dtype','--speculative-config','--enable-prompt-tokens-details',
             '--distributed-executor-backend','--language-model-only',
             '--tool-call-parser','--reasoning-parser','--enable-auto-tool-choice',
             '--default-chat-template-kwargs','--disable-uvicorn-access-log',
             '--no-enable-prefix-caching','--enforce-eager','--max-num-batched-tokens','--profiler-config'}
    absent=sorted({x for x in flags if x in special and x not in help_text})
    if absent: raise RuntimeError('UNSUPPORTED_CLI: '+repr(absent))

def gpu_inventory():
    r=subprocess.run(['nvidia-smi','--query-gpu=index,name,memory.total,memory.used',
                      '--format=csv,noheader,nounits'],capture_output=True,text=True,timeout=20)
    return {'ok':r.returncode==0,'data':r.stdout.strip(),'error':r.stderr[-800:]}

def gpu_compute_pids():
    r=subprocess.run(['nvidia-smi','--query-compute-apps=pid','--format=csv,noheader,nounits'],
                     capture_output=True,text=True,timeout=20)
    if r.returncode: return None
    return sorted({int(x.strip()) for x in r.stdout.splitlines() if x.strip().isdigit()})

def port_available(port):
    s=socket.socket(); s.settimeout(1)
    try: s.bind(('127.0.0.1',port)); return True
    except OSError: return False
    finally: s.close()

def http_json(url, payload=None, timeout=60):
    body=None if payload is None else json.dumps(payload,ensure_ascii=False).encode('utf-8')
    req=urllib.request.Request(url,data=body,headers={'Content-Type':'application/json'} if body is not None else {})
    with urllib.request.urlopen(req,timeout=timeout) as r: return json.loads(r.read().decode())

def is_ready(port):
    try:
        with urllib.request.urlopen(f'http://127.0.0.1:{port}/health',timeout=3) as r:
            if r.status!=200:return False
        d=http_json(f'http://127.0.0.1:{port}/v1/models',timeout=3)
        return any(x.get('id')=='q27' for x in d.get('data',[]))
    except Exception:return False

class ManagedServer:
    def __init__(self, name, exe, a, port, visible=None, nsys=False):
        self.name,self.a,self.exe,self.port,self.visible=name,a,P(exe),port,visible
        self.proc=None; self.log=OUTPUT/(name+'_server.log')
        self.env=cuda_env(exe,a)
        if visible is not None:self.env['CUDA_VISIBLE_DEVICES']=str(visible)
        self.flags=cli_for(a,port)
        self.nsys=nsys
        self.cmd=[str(exe),'serve',*self.flags]
        if nsys:
            n=sys.modules['shutil'].which('nsys')
            if not n: raise RuntimeError('NSIGHT_NOT_INSTALLED')
            self.cmd=[n,'profile','--trace=cuda,nvtx,osrt','--force-overwrite=true',
                      '-o',str(OUTPUT/(name+'_trace')),*self.cmd]
    def start(self):
        if not port_available(self.port): raise RuntimeError(f'PORT_BUSY_{self.port}')
        help_text=help_for(self.exe,self.env)
        validate_flags(self.flags,help_text)
        self.fh=self.log.open('w',encoding='utf-8')
        self.t0=time.monotonic()
        self.proc=subprocess.Popen(self.cmd,env=self.env,stdout=self.fh,stderr=subprocess.STDOUT,
                                   start_new_session=True)
        return self
    def wait_ready(self):
        if self.proc is None:raise RuntimeError('SERVER_NOT_STARTED')
        while time.monotonic()-self.t0<STARTUP_LIMIT_SEC:
            if self.proc.poll() is not None: raise RuntimeError('SERVER_EXIT rc='+str(self.proc.returncode))
            if is_ready(self.port):
                self.startup_s=time.monotonic()-self.t0;return self.startup_s
            time.sleep(3)
        raise RuntimeError('SERVER_STARTUP_TIMEOUT')
    def stop(self):
        if self.proc is not None and self.proc.poll() is None:
            with contextlib.suppress(Exception): os.killpg(self.proc.pid,signal.SIGTERM)
            try:self.proc.wait(timeout=45)
            except subprocess.TimeoutExpired:
                with contextlib.suppress(Exception):os.killpg(self.proc.pid,signal.SIGKILL)
                with contextlib.suppress(Exception):self.proc.wait(timeout=35)
        if hasattr(self,'fh'):self.fh.close()
        self.proc=None

def parse_prometheus(text):
    """Counter _total sums across engine labels; gauges take max; absent metrics remain absent.

    No default zero values. Prometheus can append timestamps and exemplar metadata.
    """
    vals={}
    pat=re.compile(r'^([a-zA-Z_:][a-zA-Z_0-9:]*)(?:\{[^}]*\})?\s+([-+]?(?:\d+(?:\.\d*)?|\.\d+)(?:[eE][-+]?\d+)?|NaN|Inf|-Inf)(?:\s|$)')
    gauges={'vllm:num_requests_running','vllm:num_requests_waiting','vllm:kv_cache_usage_perc'}
    for line in text.splitlines():
        if line.startswith('#'):continue
        m=pat.match(line)
        if not m:continue
        name=m.group(1)
        try:val=float(m.group(2))
        except ValueError:continue
        if not math.isfinite(val):continue
        if name in gauges:vals[name]=max(vals.get(name,float('-inf')),val)
        else:vals[name]=vals.get(name,0.0)+val
    return vals

def metrics(port):
    try:
        with urllib.request.urlopen(f'http://127.0.0.1:{port}/metrics',timeout=5) as r:
            body=r.read(3000000).decode('utf-8','replace')
        vals=parse_prometheus(body)
        required={'vllm:num_requests_running','vllm:kv_cache_usage_perc'}
        if not required.issubset(vals):vals['METRICS_PARTIAL']=sorted(required-set(vals))
        return vals
    except Exception as e:
        return {'METRICS_UNAVAILABLE':repr(e)}


## 4. Workloads + correctness: cheap first, then 40k/90k/130k

Every arm first runs **strict 3-prompt numeric content gate**, then same-token-budget concurrency load. Synthetic prompt profiles cannot certify real-agent throughput. For production conclusions provide `TRACE_JSONL` with previous exact ARC rendered prompts; no synthetic speed result will be described as ARC accuracy.

For true target-only/MTP parity a same-weights target-only reference arm must run first. We save text/first-difference; **missing logprobs or token IDs = INCONCLUSIVE**, not PASS. Long-context forced-draft rejection + Mamba state probes need engine-level instrumentation; this notebook does not claim to execute them. Source-level patch arms require separate numerical oracles before promotion.

In [ ]:
PROFILES={
  'SMOKE': {'contexts':[2048], 'streams':[4], 'max_output':128, 'repeats':1},
  'STANDARD':{'contexts':[8192,40000], 'streams':[8,16,20], 'max_output':384, 'repeats':2},
  'XL':{'contexts':[40000,90000,130000], 'streams':[8,12,16,20], 'max_output':512, 'repeats':2},
  'TRACE':{'contexts':[], 'streams':[8,16], 'max_output':512, 'repeats':1},
}
if PROFILE not in PROFILES: raise ValueError('Unknown profile')

def load_tokenizer(model):
    try:
        from transformers import AutoTokenizer
        return AutoTokenizer.from_pretrained(str(model),local_files_only=True,trust_remote_code=False)
    except Exception as e:
        raise RuntimeError('TOKENIZER_UNAVAILABLE: Cannot validate input token lengths; no fabricated context sizes. '+repr(e))

def _chat_input_tokens(tokenizer, text_or_messages, thinking=True):
    extra={'preserve_thinking':True}
    if isinstance(text_or_messages,dict):
        rec=text_or_messages
        messages=rec.get('messages') or [{'role':'user','content':rec['prompt']}]
        thinking=(rec.get('chat_template_kwargs') or {}).get('enable_thinking',thinking)
        if rec.get('tools') is not None: extra['tools']=rec['tools']
    elif isinstance(text_or_messages,list):messages=text_or_messages
    else:messages=[{'role':'user','content':text_or_messages}]
    try:
        ids=tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True,
                                           enable_thinking=thinking, **extra)
        return len(ids)
    except Exception as e:
        raise RuntimeError('CHAT_TEMPLATE_LENGTH_UNVERIFIED: '+repr(e))

def make_prompt(tokenizer,desired,variant=0):
    """Bisection over repetitions: O(log n) encodes instead of the old O(n^2) full-text retokenization loop."""
    if desired < 256: raise ValueError('Context target too short for controlled test')
    block=('Historical observation: an integer grid was transformed by rotation, recoloring, '
           'connectivity and translation. Check each hypothesis independently. ')
    intro=f'Experiment {variant}. Analyze multiple patterns in a grid transformation.\n'
    suffix=f'\nVariant={variant}. Output a numbered list of 200 distinct ideas.\n'
    def compose(n): return intro+block*n+suffix
    unit=max(1,len(tokenizer.encode(block,add_special_tokens=False)))
    hi=max(2,int(desired/unit*1.2)+8);lo=0;best=(0,_chat_input_tokens(tokenizer,compose(0)))
    while lo<=hi:
        mid=(lo+hi)//2
        count=_chat_input_tokens(tokenizer,compose(mid))
        if count<=desired:
            best=(mid,count);lo=mid+1
        else:hi=mid-1
    n,actual=best
    if actual < desired-max(128,unit*3):
        raise RuntimeError(f'PROMPT_SIZE_OFF_TARGET wanted={desired} actual={actual}')
    return compose(n),actual

def load_trace():
    if not TRACE_JSONL.is_file():raise FileNotFoundError('TRACE profile requires JSONL with prompt or messages')
    rows=[]
    for line in TRACE_JSONL.read_text(encoding='utf-8').splitlines():
        if not line.strip():continue
        r=json.loads(line)
        if not isinstance(r,dict) or not (r.get('prompt') or r.get('messages')):
            raise ValueError('TRACE row missing prompt/messages')
        rows.append(r)
    if not rows:raise ValueError('Empty TRACE JSONL')
    return rows

QUALITY_CASES=[('Reply with the single integer result of 2 plus 2.','4'),
               ('Reply with the single integer result of 7 times 6.','42'),
               ('Reply with the single integer result of 9 minus 3.','6')]

def chat_request(port, user_text, max_tokens=128, force_long=False, messages=None, stream=False, request_payload=None):
    payload={'model':'q27','messages':messages if messages is not None else [{'role':'user','content':user_text}],
             'temperature':0,'max_tokens':int(max_tokens),'stream':bool(stream),
             'seed':SEED,
             'chat_template_kwargs':{'enable_thinking':bool(force_long)}}
    if request_payload is not None:
        if not isinstance(request_payload,dict):raise TypeError('TRACE request must be dict')
        if 'messages' in request_payload:payload['messages']=request_payload['messages']
        elif 'prompt' in request_payload:payload['messages']=[{'role':'user','content':request_payload['prompt']}]
        # Preserve tool definition, constraints and relevant sampling policy; never allow hidden endpoint/model override.
        for k in ('tools','tool_choice','parallel_tool_calls','response_format','stop','top_p','top_k',
                  'temperature','seed','chat_template_kwargs'):
            if k in request_payload:payload[k]=request_payload[k]
    if stream:
        payload['stream_options']={'include_usage':True}
    t=time.monotonic()
    try:
        if stream:
            req=urllib.request.Request(f'http://127.0.0.1:{port}/v1/chat/completions',
                                       data=json.dumps(payload,ensure_ascii=False).encode('utf-8'),
                                       headers={'Content-Type':'application/json','Accept':'text/event-stream'})
            pieces=[]; reasoning=[];usage={};finish=None; first=None; call_parts={}
            with urllib.request.urlopen(req,timeout=HTTP_TIMEOUT_SEC) as response:
                for raw in response:
                    line=raw.decode('utf-8','replace').strip()
                    if not line.startswith('data:'):continue
                    data=line[5:].strip()
                    if data=='[DONE]':break
                    if not data:continue
                    chunk=json.loads(data)
                    if isinstance(chunk.get('usage'),dict):usage=chunk['usage']
                    for choice in chunk.get('choices') or []:
                        delta=choice.get('delta') or {}
                        a=delta.get('content') or ''
                        b=delta.get('reasoning_content') or ''
                        if a:pieces.append(a)
                        if b:reasoning.append(b)
                        for tc in delta.get('tool_calls') or []:
                            idx=int(tc.get('index',0))
                            entry=call_parts.setdefault(idx,{'id':'','type':'function','function':{'name':'','arguments':''}})
                            if tc.get('id'): entry['id']+=str(tc['id'])
                            fun=tc.get('function') or {}
                            if fun.get('name'):entry['function']['name']+=str(fun['name'])
                            if fun.get('arguments'):entry['function']['arguments']+=str(fun['arguments'])
                        if (a or b or delta.get('tool_calls')) and first is None:first=time.monotonic()
                        if choice.get('finish_reason'):finish=choice['finish_reason']
            tool_calls=[call_parts[k] for k in sorted(call_parts)]
            content=''.join(reasoning)+''.join(pieces)
            if tool_calls:content+='\n'+json.dumps(tool_calls,sort_keys=True,ensure_ascii=False)
            det=usage.get('prompt_tokens_details') or {}
            return {'ok':True,'content':content,'tool_calls':tool_calls,'assistant_text':''.join(pieces),
                    'reasoning_text':''.join(reasoning),'usage':usage,
                    'cached_tokens':det.get('cached_tokens'), 'first_token_s':(first-t if first else None),
                    'finish_reason':finish,'logprobs':None,
                    'elapsed':time.monotonic()-t,'error':None}
        r=http_json(f'http://127.0.0.1:{port}/v1/chat/completions',payload,timeout=HTTP_TIMEOUT_SEC)
        choice=r['choices'][0]
        msg=choice.get('message') or {}
        assistant_text=msg.get('content') or ''
        reasoning_text=msg.get('reasoning_content') or ''
        tool_calls=msg.get('tool_calls') or []
        content=reasoning_text+assistant_text
        if tool_calls:content+='\n'+json.dumps(tool_calls,sort_keys=True,ensure_ascii=False)
        usage=r.get('usage') or {}
        details=usage.get('prompt_tokens_details') or {}
        return {'ok':True,'content':content,'tool_calls':tool_calls,'assistant_text':assistant_text,'reasoning_text':reasoning_text,
                'usage':usage,'cached_tokens':details.get('cached_tokens'),
                'finish_reason':choice.get('finish_reason'),'logprobs':choice.get('logprobs'),
                'first_token_s':None,'elapsed':time.monotonic()-t,'error':None}
    except Exception as e:
        return {'ok':False,'error':repr(e),'elapsed':time.monotonic()-t,'content':'','usage':{},
                'first_token_s':None}

def quality_gate(port):
    cases=[]
    for p,exp in QUALITY_CASES:
        r=chat_request(port,p,max_tokens=40,force_long=False)
        got=r.get('content','').strip().rstrip(' .!')
        valid=(r['ok'] and got==exp and r.get('finish_reason')!='length')
        cases.append({'expected':exp,'output':got[:200],'valid':valid,'raw':r})
    return {'pass':all(x['valid'] for x in cases),'cases':cases}

def output_integrity(rows):
    # Objective failure: error/empty/repetition/short output; absent fields do not become zero.
    problems=[]
    for r in rows:
        if not r.get('ok'):
            problems.append('REQUEST_FAILED');continue
        generated=r.get('usage',{}).get('completion_tokens')
        if generated is None:problems.append('MISSING_COMPLETION_TOKENS');continue
        if generated<8:problems.append('TOO_FEW_TOKENS')
        s=r.get('content','')
        if not s.strip() and not r.get('tool_calls'):problems.append('EMPTY_TEXT')
        for tc in r.get('tool_calls') or []:
            fun=tc.get('function') or {}
            if not fun.get('name'):
                problems.append('MALFORMED_TOOL_NAME')
            try:
                args=json.loads(fun.get('arguments',''))
                if not isinstance(args,dict):problems.append('MALFORMED_TOOL_ARGUMENTS')
            except Exception:problems.append('MALFORMED_TOOL_ARGUMENTS')
        if re.search(r'(.)\1{100,}',s,re.DOTALL):problems.append('REPEATED_CHARACTER')
    return sorted(set(problems))

def benchmark_wave(ports, prompts, output_cap=256, repeat=1, synthetic=True):
    # One request per port round-robin, measuring accepted output tokens; not draft proposals.
    records=[]; before=metrics(ports[0]); t0=time.monotonic()
    stop_poll=threading.Event(); running_samples=[];kv_cache_samples=[];gpu_mem_samples=[]
    def sample_active():
        sample_idx=0
        while not stop_poll.wait(1.2):
            sample_idx+=1
            raw=[metrics(port) for port in ports]
            row=[d.get('vllm:num_requests_running') for d in raw]
            if all(isinstance(x,(float,int)) for x in row):running_samples.append(sum(row))
            pct=[d.get('vllm:gpu_cache_usage_perc') for d in raw]
            if all(isinstance(x,(float,int)) for x in pct):kv_cache_samples.append(max(pct))
            if sample_idx%2==0:
                try:
                    # nvidia-smi is an observational sample, not a memory peak guaranteed by CUDA allocator.
                    data=gpu_inventory()['data']
                    vals=[float(x.split(',')[-1].strip()) for x in data.splitlines() if x.strip()]
                    if vals:gpu_mem_samples.append(vals)
                except Exception:pass
    monitor=threading.Thread(target=sample_active,daemon=True);monitor.start()
    try:
        records=[None]*len(prompts)
        with concurrent.futures.ThreadPoolExecutor(max_workers=len(prompts)) as pool:
            fut={pool.submit(chat_request,ports[i%len(ports)],p if isinstance(p,str) else '',
                             max_tokens=output_cap,force_long=True,
                             messages=p if isinstance(p,list) else None,
                             stream=STREAM_BENCHMARK,
                             request_payload=p if isinstance(p,dict) else None):i for i,p in enumerate(prompts)}
            for f in concurrent.futures.as_completed(fut):
                records[fut[f]]=f.result()
    finally:stop_poll.set();monitor.join(timeout=3)
    wall=time.monotonic()-t0; after=metrics(ports[0])
    errors=output_integrity(records)
    if synthetic:
        for r in records:
            if r.get('ok'):
                used=r.get('usage',{}).get('completion_tokens')
                if used is None or used < output_cap*MIN_SYNTHETIC_COMPLETION_FRACTION:
                    errors.append('SHORT_SYNTHETIC_OUTPUT_NOT_COMPARABLE')
        errors=sorted(set(errors))
    num=[r.get('usage',{}).get('completion_tokens') for r in records]
    valid_num=len(num)==len(records) and all(isinstance(x,int) and x>=0 for x in num)
    total=sum(num) if valid_num else None
    # Not mathematically pure decode if requests have staggered TTFT; report explicitly as an approximation.
    ttfts=[r.get('first_token_s') for r in records]
    valid_ttft=bool(STREAM_BENCHMARK and all(isinstance(x,(int,float)) for x in ttfts))
    active_window=(max(r['elapsed'] for r in records)-min(ttfts)) if valid_ttft else None
    return {'streaming_mode':STREAM_BENCHMARK,
            'first_token_to_last_completion_tps':(total/active_window if total is not None and active_window and active_window>0 else None),
            'ttft_seconds':ttfts,'wave_kind':'MIXED_COLD_OR_WARM_INCLUDES_PREFILL',
            'wall_sec':wall,'output_tokens':total,'aggregate_tps':total/wall if total is not None and wall>0 else None,
            'requested_streams':len(prompts),'gpu_running_observed':max(running_samples) if running_samples else None,
            'gpu_running_samples':running_samples,
            'kv_cache_peak_fraction_sampled':max(kv_cache_samples) if kv_cache_samples else None,
            'gpu_peak_mib_sampled_per_card':([max(row[i] for row in gpu_mem_samples if len(row)>i)
                                              for i in range(max(map(len,gpu_mem_samples)))] if gpu_mem_samples else None),
            'metrics_pre':before,'metrics_post':after,'running_samples':running_samples,
            'failed_requests':sum(not r['ok'] for r in records),'problems':errors,
            'prompt_tokens_sum':sum(x.get('usage',{}).get('prompt_tokens',0) for x in records if x.get('ok')),
            'cached_tokens_sum':sum(x['cached_tokens'] for x in records if isinstance(x.get('cached_tokens'),int)) if all(isinstance(x.get('cached_tokens'),int) for x in records) else None,
            'responses':[{'ok':x['ok'],'error':x.get('error'),'usage':x.get('usage'),
                          'cached_tokens':x.get('cached_tokens'),'finish_reason':x.get('finish_reason'),
                          'tool_calls':x.get('tool_calls'),
                          'sha256':hashlib.sha256(x.get('content','').encode()).hexdigest(),
                          'completion_tokens':x.get('usage',{}).get('completion_tokens'),
                          'prefix':x.get('content','')[:180]} for x in records]}

def repeated_prefix_probe(port):
    # Same request then appended tool message tests cache retention. Actual tool-use trace is better.
    long='Compute a grid transformation carefully. '+('Check diagonal shapes, adjacency, and color count. '*160)
    msgs=[{'role':'user','content':long}]
    r1=chat_request(port,'',max_tokens=96,force_long=False,messages=msgs)
    r2=chat_request(port,'',max_tokens=96,force_long=False,messages=msgs)
    msgs2=msgs+[{'role':'assistant','content':r1.get('assistant_text','')},
                {'role':'user','content':'Tool returned a tiny 3x3 grid. Continue the prior analysis.'}]
    r3=chat_request(port,'',max_tokens=96,force_long=False,messages=msgs2)
    return {'cached_tokens':[x.get('cached_tokens') for x in [r1,r2,r3]],
            'output_sha256':[hashlib.sha256(x.get('content','').encode()).hexdigest() for x in [r1,r2,r3]],
            'all_requests_ok':all(x['ok'] for x in (r1,r2,r3)),
            'generated_assistant_reply_reuse_verified':False,
            'not_compared_if_missing_cache':any(x.get('cached_tokens') is None for x in (r1,r2,r3))}
if MODE=='RUN' and PROFILE=='XL' and MODEL_LEN < 130000+PROFILES['XL']['max_output']:
    raise ValueError('XL profile requires Q27_MODEL_LEN >= 130000 + max_output. Set model_len with capacity proof before GPU spend.')


## 4A. Same-weight exact-token oracle এবং prefill-settled steady decode
Target-only eager arms **V0_FP8_ORACLE_EAGER** ও **V0_INT4_ORACLE_EAGER** আগে চালান।
সুস্পষ্টভাবে আলাদা: `wave_output_tps`, `steady_decode_tps`, এবং `agent_trace_tps`।
প্রতিটি oracle current session-scoped; guessed tokens বা missing metrics দিয়ে PASS নয়।

In [ ]:
# CONSOLIDATED v3: exact-token oracle and sampled steady-decode counter-window.
# Reference approach: second AUDITED_v2 notebook, source-specific corrections and fail-closed gating.
ORACLE_TOKEN_ROWS={}
ORACLE_EXPERIMENT_BANDS=[b for b in ORACLE_BANDS if b+256<MODEL_LEN]
ORACLE_SMOKE_TEXT=[
    'Calculate 2+2. Answer in one sentence.',
    'Repeat exactly: ARC-GRID-17.',
    'Given input grid [0,1;1,0] and output [1,0;0,1], describe a Python transform.',
]

def checkpoint_identity(model_path):
    root=P(model_path)
    ok,why=model_shards_ok(root)
    if not ok:raise RuntimeError('MODEL_IDENTITY_SHARDS_INVALID '+str(why))
    files=[root/'config.json']
    files+=sorted(root.glob('*.safetensors.index.json'))
    files+=[root/k for k in ('tokenizer.json','tokenizer.model','tokenizer_config.json') if (root/k).is_file()]
    if not any(p.name in ('tokenizer.json','tokenizer.model') for p in files):
        raise RuntimeError('TOKENIZER_IDENTITY_MISSING')
    return {'realpath':str(root.resolve()),'manifest_hash':hashlib.sha256(
        json.dumps([(p.name,hashlib.sha256(p.read_bytes()).hexdigest()) for p in files],sort_keys=True).encode()
    ).hexdigest()}

class TokenStream(threading.Thread):
    """Raw /v1/completions SSE. Missing/inconsistent token_ids => inconclusive, never synthesized from text."""
    def __init__(self,port,ids,max_new=160,temperature=0.0,logprobs=2):
        super().__init__(daemon=True)
        self.port,self.ids,self.max_new,self.temperature,self.logprobs=port,list(ids),int(max_new),temperature,logprobs
        self.tokens=[];self.margins=[];self.err=None;self.done=False;self.finish=None
        self.first_t=None;self.submit_t=None;self.end_t=None;self.response=None;self.cancelled=False
    def run(self):
        self.submit_t=time.monotonic()
        body={'model':'q27','prompt':self.ids,'max_tokens':self.max_new,'temperature':self.temperature,
              'ignore_eos':True,'return_token_ids':True,'stream':True,'seed':SEED}
        if self.logprobs:body['logprobs']=int(self.logprobs)
        try:
            request=urllib.request.Request(f'http://127.0.0.1:{self.port}/v1/completions',
                 data=json.dumps(body).encode(),headers={'Content-Type':'application/json','Accept':'text/event-stream'},method='POST')
            self.response=urllib.request.urlopen(request,timeout=max(HTTP_TIMEOUT_SEC,300))
            for raw in self.response:
                if self.cancelled:break
                if not raw.lstrip().startswith(b'data:'):continue
                msg=raw.split(b'data:',1)[1].strip()
                if msg==b'[DONE]':self.done=True;break
                if not msg:continue
                try:obj=json.loads(msg)
                except Exception as e:self.err='MALFORMED_SSE_JSON '+repr(e);break
                if obj.get('error') or obj.get('object')=='error':
                    self.err='ENGINE_SSE_ERROR '+str(obj.get('error'))[:240];break
                for ch in obj.get('choices') or []:
                    toks=ch.get('token_ids')
                    if toks is None:
                        self.err='MISSING_TOKEN_IDS';break
                    if not isinstance(toks,list) or any(type(k)!=int for k in toks):
                        self.err='INVALID_TOKEN_IDS';break
                    # Do not infer/reconstruct tokens from streamed text.
                    if toks and self.first_t is None:self.first_t=time.monotonic()
                    self.tokens.extend(toks)
                    for row in (ch.get('logprobs') or {}).get('top_logprobs') or []:
                        ys=sorted((v for v in (row or {}).values() if isinstance(v,(int,float))),reverse=True)
                        self.margins.append(ys[0]-ys[1] if len(ys)>1 else None)
                    if ch.get('finish_reason') is not None:self.finish=ch['finish_reason']
                if self.err:break
        except Exception as e:
            if not self.cancelled:self.err=type(e).__name__+': '+str(e)[:250]
        finally:
            self.end_t=time.monotonic()
            with contextlib.suppress(Exception):
                if self.response is not None:self.response.close()
            if not self.cancelled and not self.err and (not self.done or self.finish is None):
                self.err='INCOMPLETE_STREAM'
    def cancel(self):
        self.cancelled=True
        # Timeout-bounded network close prevents indefinite run-arm blockage.
        try:
            raw=getattr(getattr(self.response,'fp',None),'raw',None)
            sock=getattr(raw,'_sock',None)
            if sock is not None:sock.shutdown(socket.SHUT_RDWR)
        except Exception:pass
    def summary(self,with_tokens=True):
        return {'tokens':self.tokens[:] if with_tokens else len(self.tokens),
            'margins':self.margins[:] if with_tokens else None,
            'error':self.err,'done':self.done,'finish_reason':self.finish,
            'first_token_s':self.first_t-self.submit_t if self.first_t and self.submit_t else None,
            'request_elapsed_s':self.end_t-self.submit_t if self.end_t and self.submit_t else None}

def token_prompt(tok,desired,variant=0,kind='shared'):
    """Token-id-level context count, shared-vs-distinct prefix; identical across same tokenizer."""
    if desired<128:raise ValueError('CONTEXT_TOO_SHORT')
    head=f'ARC analysis study {variant}\n' if kind=='distinct' else 'ARC analysis shared history\n'
    unit=tok.encode('Grid 0 1; rotation reflection color rule; compare all examples.\n',add_special_tokens=False)
    begin=tok.encode(head,add_special_tokens=False)
    tail=tok.encode(f'\nTask variant {variant}. Output a rigorous answer.',add_special_tokens=False)
    if not unit or len(begin)+len(tail)>=desired:raise ValueError('INVALID_PROMPT_UNIT')
    keep=desired-len(begin)-len(tail)
    ids=begin+(unit*(keep//len(unit)+1))[:keep]+tail
    assert len(ids)==desired
    return ids

def run_token_probe(port,tok,band=2048,variant=0,max_new=160,timeout=390):
    g=TokenStream(port,token_prompt(tok,band,variant),max_new=max_new,temperature=0,logprobs=2)
    g.start();g.join(timeout=timeout)
    if g.is_alive():g.cancel();g.join(timeout=8)
    s=g.summary()
    if g.is_alive():s['error']='STUCK_TOKEN_CLIENT'
    return {'band':band,**s}

def assess_token_rows(ref,candidate):
    if len(ref)!=len(candidate) or not ref:return {'status':'INCONCLUSIVE_ORACLE_PAIRS_MISSING'}
    differences=[]
    for x,y in zip(ref,candidate):
        if x['band']!=y['band'] or x.get('error') or y.get('error') or not x.get('done') or not y.get('done'):
            return {'status':'INCONCLUSIVE_TOKEN_CLIENT', 'first_failure_band':x.get('band')}
        a,b=x['tokens'],y['tokens']
        if not a or not b:return {'status':'INCONCLUSIVE_NO_TOKENS'}
        if a==b and x['finish_reason']==y['finish_reason']:continue
        diff=next((i for i in range(min(len(a),len(b))) if a[i]!=b[i]),None)
        if diff is None:return {'status':'INCONCLUSIVE_LENGTH_OR_STOP_MISMATCH','band':x['band']}
        margin=x.get('margins') or []
        m=margin[diff] if diff<len(margin) else None
        differences.append({'band':x['band'],'first_diff_token':diff,'oracle_margin':m})
        if m is None:return {'status':'INCONCLUSIVE_MARGIN_NOT_AVAILABLE','differences':differences}
        if m>=0.5:return {'status':'FAIL_HIGH_MARGIN_TOKEN_DIVERGENCE','differences':differences}
    return {'status':'TOKEN_ORACLE_EXACT' if not differences else 'INCONCLUSIVE_NEAR_TIE','differences':differences}

def execute_token_oracle(a,port,tok):
    bands=[2048]+[b for b in ORACLE_EXPERIMENT_BANDS if b!=2048]
    if PROFILE=='SMOKE':bands=[2048]
    bands=[b for b in bands if b+256 < int(a.get('max_len',MODEL_LEN))]
    if not bands:return {'status':'INCONCLUSIVE_NO_ORACLE_BANDS'}
    rows=[]
    for b in bands:
        r=run_token_probe(port,tok,band=b)
        rows.append(r)
        if r.get('error') or not r.get('tokens') or not r.get('done'):
            return {'status':'INCONCLUSIVE_ORACLE_CAPTURE_FAILED','rows':rows}
    ident=checkpoint_identity(choose_model(a))
    model=a['weight']
    if a['track']=='oracle':
        ORACLE_TOKEN_ROWS[model]={'rows':rows,'identity':ident,'source_arm':a['name']}
        return {'status':'ORACLE_SAVED_IN_CURRENT_SESSION','rows':rows,'identity':ident}
    ref=ORACLE_TOKEN_ROWS.get(model)
    if ref is None:return {'status':'INCONCLUSIVE_NO_SAME_WEIGHT_ORACLE','rows':rows,'identity':ident}
    if ref['identity']!=ident:return {'status':'INCONCLUSIVE_MODEL_IDENTITY_MISMATCH','rows':rows}
    # The exact same band layout must match; no silent partial comparison.
    return {**assess_token_rows(ref['rows'],rows),'rows':rows}

_STEADY_STATES={'STEADY_MEASURED','CAPACITY_LIMITED_STEADY_MEASURED'}
def steady_decode_window(port,tok,band,requested,seconds=None,max_new=8192):
    """Measure accepted engine output tokens only after every *active* request has emitted a token.
    Distinguish capacity-limited from full-concurrency; missing telemetry never coerced to zero.
    """
    seconds=seconds or STEADY_WINDOW_SECONDS
    r={'context_target':band,'requested_streams':requested,'status':'INCONCLUSIVE_NOT_STARTED'}
    if band+max_new+128 >= MODEL_LEN:
        max_new=min(max_new,MODEL_LEN-band-192)
    if max_new<512:return {**r,'status':'SKIPPED_STEADY_CONTEXT_LIMIT'}
    ids=[token_prompt(tok,band,i,'shared') for i in range(requested)]
    streams=[TokenStream(port,item,max_new=max_new,temperature=0.7,logprobs=0) for item in ids]
    def get():return metrics(port)
    base=get()
    if 'vllm:generation_tokens_total' not in base:return {**r,'status':'INCONCLUSIVE_GENERATION_COUNTER_MISSING'}
    try:
        for g in streams:g.start()
        deadline=time.monotonic()+STEADY_ADMISSION_SECONDS
        best=0;prev_prompt=None;settled=0;stable_samples=[]
        while time.monotonic()<deadline:
            m=get()
            required=['vllm:num_requests_running','vllm:num_requests_waiting',
                      'vllm:prompt_tokens_total','vllm:generation_tokens_total']
            if any(k not in m for k in required):
                r['status']='INCONCLUSIVE_PREFILL_METRIC_MISSING';return r
            active=int(m['vllm:num_requests_running']);waiting=int(m['vllm:num_requests_waiting']);best=max(best,active)
            first=sum(g.first_t is not None for g in streams)
            if active>0 and first>=active and m['vllm:prompt_tokens_total']==prev_prompt:
                settled+=1
            else:settled=0
            prev_prompt=m['vllm:prompt_tokens_total']
            stable_samples.append({'running':active,'waiting':waiting,'first_tokens':first})
            if settled>=3:break
            if all(not g.is_alive() for g in streams):
                r['status']='INCONCLUSIVE_OUTPUT_COMPLETED_BEFORE_SETTLED';return r
            time.sleep(1.5)
        if settled<3 or best<1:return {**r,'status':'INCONCLUSIVE_NO_DECODE_SETTLEMENT','admission':stable_samples[-8:]}
        m0=get();t0=time.monotonic();p0=m0.get('vllm:num_preemptions_total')
        if 'vllm:generation_tokens_total' not in m0:return {**r,'status':'INCONCLUSIVE_GENERATION_COUNTER_MISSING'}
        measurements=[]
        while time.monotonic()-t0<seconds:
            time.sleep(min(1.5,seconds-(time.monotonic()-t0)))
            m=get();measurements.append(m)
            if m.get('METRICS_UNAVAILABLE') or 'vllm:num_requests_running' not in m:
                return {**r,'status':'INCONCLUSIVE_MISSING_WINDOW_METRICS'}
            if all(not g.is_alive() for g in streams):
                return {**r,'status':'INCONCLUSIVE_STREAMS_DONE_DURING_WINDOW'}
        m1=get();t1=time.monotonic()
        if 'vllm:generation_tokens_total' not in m1:return {**r,'status':'INCONCLUSIVE_GENERATION_COUNTER_MISSING'}
        prefill0=m0.get('vllm:prompt_tokens_total')
        if prefill0 is None or any(m.get('vllm:prompt_tokens_total') != prefill0 for m in measurements+[m1]):
            return {**r,'status':'INCONCLUSIVE_PREFILL_DURING_DECODE_WINDOW'}
        count=m1['vllm:generation_tokens_total']-m0['vllm:generation_tokens_total']
        run=[m['vllm:num_requests_running'] for m in measurements]
        if count<=0:return {**r,'status':'INCONCLUSIVE_NO_DECODE_PROGRESS'}
        if p0 is not None and m1.get('vllm:num_preemptions_total',p0)>p0:
            return {**r,'status':'INCONCLUSIVE_PREEMPTION_DETECTED'}
        if min(run,default=0)<0.8*best:
            return {**r,'status':'INCONCLUSIVE_RUNNING_NOT_STABLE','running':run}
        r.update(status='CAPACITY_LIMITED_STEADY_MEASURED' if best<requested else 'STEADY_MEASURED',
            steady_decode_tps=count/(t1-t0),steady_output_tokens=int(count),steady_seconds=t1-t0,
            actual_running_max=best,actual_running_mean=statistics.mean(run) if run else None,
            kv_utilization_peak=max((m.get('vllm:kv_cache_usage_perc') for m in measurements
                                     if m.get('vllm:kv_cache_usage_perc') is not None),default=None),
            begin_monotonic=t0,end_monotonic=t1)
        return r
    finally:
        for g in streams:g.cancel()
        for g in streams:
            if g.ident is not None:g.join(timeout=7)
        if any(g.is_alive() for g in streams):
            r['status']='FAIL_STEADY_CLIENT_LEAK'
        idle_deadline=time.monotonic()+25;idle=False
        while time.monotonic()<idle_deadline:
            m=get()
            if m.get('vllm:num_requests_running')==0 and m.get('vllm:num_requests_waiting')==0:
                idle=True;break
            time.sleep(1)
        if not idle:r['status']='FAIL_STEADY_SERVER_NOT_IDLE'

# CPU-only independent regression cases; no GPU/network/model files required.
def consolidate_cpu_regressions():
    assert len(ARMS)>=52, len(ARMS)
    for v in ('V0_FP8_ORACLE_EAGER','V0_INT4_ORACLE_EAGER','V11_TP1X2_REPLICAS',
              'V12_HOST_STAGED_AR','V9_NVFP4_KV_MTP3','V0_PROFILE_TORCH'):
        assert v in ARMS
    assert '--disable-custom-all-reduce' not in cli_for(ARMS['V12_HOST_STAGED_AR'],19110)
    assert '--disable-custom-all-reduce' in cli_for(ARMS['V0_FP8_BASE'],19110)
    assert '--enforce-eager' in cli_for(ARMS['V0_FP8_ORACLE_EAGER'],19110)
    assert '--no-enable-prefix-caching' in cli_for(ARMS['V0_FP8_ORACLE_EAGER'],19110)
    assert '--max-num-batched-tokens' in cli_for(ARMS['V13_PREFILL2048'],19110)
    m=parse_prometheus("""# HELP x x\nvllm:kv_cache_usage_perc{model="q27"} 0.82\nvllm:num_requests_running 7\nvllm:generation_tokens_total{model="q27"} 812\nvllm:prompt_tokens_total 16000\n""")
    assert m['vllm:kv_cache_usage_perc']==0.82
    assert m['vllm:generation_tokens_total']==812
    assert 'vllm:generation_tokens_total' not in parse_prometheus('vllm:num_requests_running 3\n')
    ref=[{'band':2048,'tokens':[1,2,3],'margins':[1.0,1.0,1.0],'error':None,'done':True,'finish_reason':'length'}]
    assert assess_token_rows(ref,ref)['status']=='TOKEN_ORACLE_EXACT'
    cand=[{**ref[0],'tokens':[1,9,3]}]
    assert assess_token_rows(ref,cand)['status']=='FAIL_HIGH_MARGIN_TOKEN_DIVERGENCE'
    assert assess_token_rows(ref,[{**cand[0],'margins':[]}])['status']=='FAIL_HIGH_MARGIN_TOKEN_DIVERGENCE'
    assert assess_token_rows(ref,[{**ref[0],'tokens':[]}])['status']=='INCONCLUSIVE_NO_TOKENS'
    # Manifest incomplete reference must always be rejected.
    import tempfile
    with tempfile.TemporaryDirectory() as root:
        d=P(root);(d/'config.json').write_text('{}');(d/'model.safetensors.index.json').write_text(
           json.dumps({'weight_map':{'x':'absent.safetensors'}}))
        assert model_shards_ok(d)[0] is False
    return {'status':'PASS','checks':['all_arm_preserved','no_custom_ar_disabled','oracle_eager',
       'v1_kv_metric','missing_generation_counter','numerical_high_margin','missing_shards']}
print('CONSOLIDATED_CPU_TESTS',consolidate_cpu_regressions())


## 5. Execution — independent servers, cold startup, bounded warmup, metrics and reports

**Run order:** `V0_FP8_TARGET_ONLY` + `V0_FP8_BASE` first for numerical/quality reference; then `V1_*`,`V2_*` etc. For INT4 compare with its own target-only oracle; cross-quantization token exactness not guaranteed। New model/quantized KV/parallelism/patch arms must pass launch + quality before throughput। No auto retry to a different configuration under the same arm name.

**CLI/error transparency:** each arm `PASS_BASIC_GATE`, `NO_GO_CORRUPT`, `INCONCLUSIVE_NO_ORACLE`, `SKIPPED_REQUIRED_ARTIFACT`, `LAUNCH_FAILED`, `BUDGET_EXHAUSTED` or `BENCH_FAILED`. `PASS_BASIC_GATE` is not a hidden ARC quality GO.

In [ ]:
def atomic_json(path,data):
    p=P(path);tmp=p.with_suffix(p.suffix+'.tmp')
    tmp.write_text(json.dumps(data,indent=2,ensure_ascii=False,default=str),encoding='utf-8');tmp.replace(p)

def possible_oracle(a, quality, save=None):
    # target-only reference is per exact checkpoint+backend, not across INT4/FP8.
    modelkey=a.get('weight','fp8')
    if not a.get('spec') and a.get('spec')!= 'external':
        return 'ORACLE_RECORDED'
    path=OUTPUT/('oracle_'+modelkey+'.json')
    if not path.exists():return 'INCONCLUSIVE_NO_TARGET_ONLY_ORACLE'
    ref=json.loads(path.read_text())
    if not ref.get('pass'):return 'INCONCLUSIVE_INVALID_ORACLE'
    return 'BASIC_GATE_WITH_ORACLE_AVAILABLE_NOT_LONG_PARITY'

# Diagnostic greedy text parity is a screening tool, not proof of equal target logits or ARC accuracy.
def oracle_fingerprint(a, port):
    prompt='Analyze a 3x3 grid with blue cells at (0,0), (1,1), and (2,2). State the transformation and list checks.'
    r=chat_request(port,prompt,max_tokens=128,force_long=False)
    if not r.get('ok') or not r.get('content','').strip():
        raise RuntimeError('ORACLE_PROBE_REQUEST_FAILED')
    return {'content_sha256':hashlib.sha256(r['content'].encode()).hexdigest(),
            'content_prefix':r['content'][:100],
            'completion_tokens':r.get('usage',{}).get('completion_tokens'),
            'finish_reason':r.get('finish_reason')}

def oracle_key(a):
    # Different weight checkpoint versions cannot be accidentally paired.
    cfg=choose_model(a)/'config.json'
    return SESSION_ID+'_'+a.get('weight','fp8')+'_'+hashlib.sha256(cfg.read_bytes()).hexdigest()[:16]

def check_oracle(a, probe):
    target=a.get('spec')==0 and not a.get('custom_ar') and not a.get('graph')
    path=OUTPUT/('target_oracle_'+oracle_key(a)+'.json')
    if target:
        atomic_json(path,probe)
        return 'TARGET_ONLY_ORACLE_SAVED'
    if not path.exists():
        return 'INCONCLUSIVE_NO_TARGET_ONLY_ORACLE'
    ref=json.loads(path.read_text())
    if ref.get('content_sha256')!=probe['content_sha256']:
        return 'DIFFERENT_GREEDY_TEXT_REQUIRES_LOGIT_REVIEW'
    return 'SHORT_TEXT_MATCH_ONLY_NOT_LONG_PARITY'

def write_summary_csv(reports):
    rows=[]
    for name,r in reports.items():
        for b in r.get('benchmarks',[]):
            rows.append({'arm':name,'status':r.get('status'),'profile':PROFILE,'context_target':b.get('target_context'),
                'clients_requested':b.get('requested_streams'),'gpu_running_observed':b.get('gpu_running_observed'),
                'output_tokens':b.get('output_tokens'),'wall_sec':b.get('wall_sec'),
                'aggregate_tps':b.get('aggregate_tps'),'errors':';'.join(b.get('problems',[])),
                'cached_tokens':b.get('cached_tokens_sum')})
    if rows:
        with (OUTPUT/'summary.csv').open('w',newline='',encoding='utf-8') as f:
            w=csv.DictWriter(f,fieldnames=list(rows[0]));w.writeheader();w.writerows(rows)

def single_arm_run(name,baseline_exe,tokenizer_cache,start_time):
    a=ARMS[name]
    rec={'arm':name,'experiment_track':a['track'],'arm_config':a,'status':'NOT_RUN',
         'notebook_revision':'AUDITED_v5',
         'started_utc':datetime.now(timezone.utc).isoformat(),'gpu_before':gpu_inventory(),
         'benchmarks':[],'correctness':{}}
    if a['track']!='nccl_micro' and (choose_model(a)/'config.json').is_file():
        rec['model_config_sha256']=hashlib.sha256((choose_model(a)/'config.json').read_bytes()).hexdigest()
    if not preflight[name]['artifact_ok']:
        rec.update(status='SKIPPED_REQUIRED_ARTIFACT',reason=preflight[name]['detail']);return rec
    if time.monotonic()-start_time>=MAX_LAB_MINUTES*60-STARTUP_LIMIT_SEC:
        rec.update(status='BUDGET_EXHAUSTED',reason='Not enough total lab time for another startup');return rec
    if a['track']=='nccl_micro':
        # This microbench uses the attached `nccl-tests` binary in single-process `-g 4` mode. Not a model result.
        try:
            out=subprocess.run([str(NCCL_TEST_BIN),'-b','262144','-e','2097152','-f','2','-g','4'],
                               capture_output=True,text=True,timeout=180)
            rec.update(status='MICROBENCH_ONLY' if out.returncode==0 else 'MICROBENCH_FAILED',
                       stdout=out.stdout[-15000:],stderr=out.stderr[-3000:])
        except Exception as e:rec.update(status='MICROBENCH_FAILED',reason=repr(e))
        return rec
    servers=[]
    try:
        replicas=int(a.get('replicas',1))
        if replicas not in (1,2,4):raise RuntimeError('Unsupported replica count')
        exe=exe_for(a,baseline_exe)
        if a.get('replicas',1)>1:
            one=REPORTS.get('V11_TP1_SINGLE',{})
            valid=any((b.get('aggregate_tps') or 0)>0 and not b.get('problems') for b in one.get('benchmarks',[]))
            if not valid or one.get('gpu_release_confirmed') is not True:
                raise RuntimeError('MERLIN_TP1_SINGLE_FEASIBILITY_GATE_FAILED')
        # Fail closed on occupied port, unintended GPUs or 4 L4 unavailable.
        if len(gpu_inventory().get('data','').splitlines())<4:
            raise RuntimeError('REQUIRE_FOUR_GPUS')
        pre_pids=gpu_compute_pids()
        rec['gpu_compute_pids_before']=pre_pids
        if pre_pids is None: raise RuntimeError('GPU_PID_QUERY_FAILED')
        # Kernel already running may use GPUs: abort rather than killing a stranger's job.
        if pre_pids and (set(pre_pids)-{os.getpid()}):
            raise RuntimeError('GPU_ALREADY_IN_USE: '+repr(pre_pids))
        for i in range(replicas):
            v=str(i) if replicas>1 else a.get('visible')
            server=ManagedServer(name+('_rep'+str(i) if replicas>1 else ''),exe,a,BASE_PORT+i,
                                 visible=v,nsys=bool(a.get('nsys')))
            servers.append(server)
            server.start()  # each replica is launched before any wait: parallel model loading
        for server in servers:
            startup=server.wait_ready()
            rec.setdefault('startup_seconds',[]).append(round(startup,2))
        ports=[s.port for s in servers]
        if a['requirement'].startswith('patch_'):
            manifest=json.loads(P(artifact_for(a)['manifest']).read_text())
            regex=manifest['activation_regex']
            matched=[]
            for server in servers:
                log=server.log.read_text(encoding='utf-8',errors='replace')
                matched.append(bool(re.search(regex,log,re.I)))
            rec['patch_activation_evidence']={'regex':regex,'observed_each_server':matched}
            if not all(matched):
                rec['patch_activation_boot']='NOT_YET_PROVEN_AT_STARTUP'
                # A patched kernel may first execute on actual decode, inspect logs after waves too.
        ports=[s.port for s in servers]
        rec['resolved_launch']=[{'cmd':s.cmd,'env_override':a.get('env',{}),'port':s.port,'log':str(s.log)} for s in servers]
        if a.get('torch_profiler'):
            d=OUTPUT/(a['name']+'_torch_traces'); d.mkdir(parents=True,exist_ok=True)
            r=urllib.request.Request(f'http://127.0.0.1:{ports[0]}/start_profile',
              data=json.dumps({'profile_prefix':a['name'],'max_iterations':60}).encode(),
              headers={'Content-Type':'application/json'},method='POST')
            with urllib.request.urlopen(r,timeout=20) as response:response.read(2000)
            rec['torch_profiler_started']=True
        q=quality_gate(ports[0]);rec['correctness']['arithmetic_3']=q
        if not q['pass']:
            rec.update(status='NO_GO_CORRUPT',reason='3 exact numeric checks fail. No throughput promotion.');return rec
        # The old arithmetic-only oracle was removed: only the actual text fingerprint is saved.
        # Require a genuine target-only fingerprint to call any arm screen-verified.
        probe=oracle_fingerprint(a,ports[0])
        rec['correctness']['oracle_probe']=probe
        rec['correctness']['oracle_status']=check_oracle(a,probe)
        rec['prefix_probe']=repeated_prefix_probe(ports[0])
        if not rec['prefix_probe']['all_requests_ok']:
            rec.update(status='INCONCLUSIVE_PREFIX_PROBE_FAILED');return rec
        tok=tokenizer_cache.get(str(choose_model(a)))
        if tok is None:
            tok=load_tokenizer(choose_model(a));tokenizer_cache[str(choose_model(a))]=tok
        rec['correctness']['token_oracle']=execute_token_oracle(a,ports[0],tok)
        if rec['correctness']['token_oracle']['status'].startswith('FAIL'):
            rec.update(status='NO_GO_TOKEN_DIVERGENCE',reason='Same-weight exact-token oracle failed')
            return rec
        # Only a separate, model-identical frozen eager control supports numerical comparison.
        pconf=PROFILES[PROFILE]
        if PROFILE=='TRACE':
            traces=load_trace(); profile_cases=[('TRACE',traces)]
        else:profile_cases=[(context,None) for context in pconf['contexts']]
        cutoff=time.monotonic()+MAX_ARM_MINUTES*60
        for ctx, traces in profile_cases:
            for clients in pconf['streams']:
                if time.monotonic()>cutoff:rec['budget_truncated']=True;break
                if traces is None:
                    # Generate long token-counted synthetic prompt; requested and effective context differ slightly due to chat template.
                    prompt,actual=make_prompt(tok,ctx,0)
                    if actual+pconf['max_output']>int(a.get('max_len',MODEL_LEN)):
                        rec['benchmarks'].append({'target_context':ctx,'requested_streams':clients,'status':'SKIPPED_CONTEXT_EXCEEDS_MODEL_LEN'});continue
                    prompts=[prompt+'\nSequence ID: %d. Output detailed ideas.'%i for i in range(clients)]
                    # The actual submitted chat template, suffix and generation reserve all count.
                    actual_submitted=max(_chat_input_tokens(tok,p) for p in prompts)
                    if actual_submitted+pconf['max_output']>int(a.get('max_len',MODEL_LEN)):
                        rec['benchmarks'].append({'target_context':ctx,'requested_streams':clients,
                                                 'status':'SKIPPED_SUBMITTED_TOKENS_EXCEED_MODEL_LEN',
                                                 'submitted_prompt_tokens':actual_submitted});continue
                else:
                    prompts=[r for r in traces[:clients]]  # retain tools and per-record chat policy; snapshot replay only
                    if len(prompts)!=clients:
                        rec['benchmarks'].append({'target_context':ctx,'requested_streams':clients,
                                                 'status':'SKIPPED_NOT_ENOUGH_TRACE_REQUESTS'});continue
                    actual=max(_chat_input_tokens(tok,p) for p in prompts)
                    if actual+pconf['max_output']>int(a.get('max_len',MODEL_LEN)):
                        rec['benchmarks'].append({'target_context':ctx,'requested_streams':clients,
                                                 'status':'SKIPPED_TRACE_EXCEEDS_MODEL_LEN','submitted_prompt_tokens':actual});continue
                for repeat in range(pconf['repeats']):
                    result=benchmark_wave(ports,prompts,pconf['max_output'],synthetic=(traces is None))
                    result.update(target_context=ctx,approx_input_tokens=actual,repeat=repeat,
                                  cache_condition=('COLD' if repeat==0 else 'WARM_REPEAT'),
                                  workload='TRACE' if traces else 'SYNTHETIC')
                    rec['benchmarks'].append(result)
                    atomic_json(OUTPUT/(name+'_live.json'),rec)
                    if ENABLE_STEADY_WINDOWS and PROFILE in ('STANDARD','XL') and repeat==pconf['repeats']-1:
                        # For replicas, each server gets a separate window; align durations before aggregation.
                        if len(ports)>1:
                            with concurrent.futures.ThreadPoolExecutor(max_workers=len(ports)) as pool:
                                fs=[pool.submit(steady_decode_window,p,tok,ctx,max(1,clients//len(ports))) for p in ports]
                                sr=[f.result() for f in fs]
                            good=all(z.get('status') in _STEADY_STATES for z in sr)
                            overlap=(min(z['end_monotonic'] for z in sr)-max(z['begin_monotonic'] for z in sr)) if good else 0
                            aligned=(good and overlap>0 and overlap/min(z['steady_seconds'] for z in sr)>0.80)
                            result['steady']={'replicas':sr,'status':'REPLICAS_ALIGNED_STEADY' if aligned else 'INCONCLUSIVE_REPLICA_WINDOWS',
                                              'aggregate_decode_tps':sum(z['steady_decode_tps'] for z in sr) if aligned else None}
                        else:
                            result['steady']=steady_decode_window(ports[0],tok,ctx,clients)
                    if result['problems'] or result['aggregate_tps'] is None:
                        rec.update(status='BENCH_FAILED',reason=repr(result['problems']));return rec
            if time.monotonic()>cutoff:break
        if a['requirement'].startswith('patch_'):
            mf=json.loads(P(artifact_for(a)['manifest']).read_text())
            rx=mf['activation_regex']
            evidence=[bool(re.search(rx,s.log.read_text(encoding='utf-8',errors='replace'),re.I)) for s in servers]
            rec['patch_activation_evidence']['observed_each_server_after_bench']=evidence
            if not all(evidence):
                rec.update(status='INCONCLUSIVE_PATCH_NOT_ACTIVATED',reason='No kernel/graph/collective activation in runtime log')
                return rec
        if not rec['benchmarks'] or not any(b.get('aggregate_tps') is not None for b in rec['benchmarks']):
            rec.update(status='INCONCLUSIVE_NO_BENCHMARK')
        elif rec.get('budget_truncated') or any(b.get('status','').startswith(('SKIPPED','INCONCLUSIVE','FAIL')) for b in rec['benchmarks']):
            rec.update(status='INCONCLUSIVE_PARTIAL_PROFILE')
        else:
            token_status=rec['correctness'].get('token_oracle',{}).get('status','')
            if any(b.get('steady',{}).get('status','') not in ('',*sorted(_STEADY_STATES),'REPLICAS_ALIGNED_STEADY') for b in rec['benchmarks']):
                rec['status']='INCONCLUSIVE_STEADY_WINDOW_FAILURE'
                return rec
            if token_status not in ('TOKEN_ORACLE_EXACT','ORACLE_SAVED_IN_CURRENT_SESSION'):
                rec['status']='DIAGNOSTIC_ONLY_UNCERTIFIED_ORACLE_'+token_status
                return rec
            oracle=rec['correctness'].get('oracle_status','')
            rec['status']=('DIAGNOSTIC_TOKEN_ORACLE_SCREEN_ONLY' if
                           oracle in ('SHORT_TEXT_MATCH_ONLY_NOT_LONG_PARITY','TARGET_ONLY_ORACLE_SAVED')
                           else 'INCONCLUSIVE_NUMERICAL_ORACLE')
        return rec
    except Exception as e:
        rec.update(status=('LAUNCH_FAILED' if not rec.get('startup_seconds') else 'BENCH_FAILED'),reason=repr(e))
        return rec
    finally:
        if rec.get('torch_profiler_started') and servers and servers[0].proc and servers[0].proc.poll() is None:
            try:
                r=urllib.request.Request(f'http://127.0.0.1:{servers[0].port}/stop_profile',
                    data=b'{}',headers={'Content-Type':'application/json'},method='POST')
                with urllib.request.urlopen(r,timeout=90) as response:response.read(2000)
                folder=OUTPUT/(a['name']+'_torch_traces')
                files=[str(x) for x in folder.rglob('*') if x.is_file()]
                rec['torch_profiler_files']=files[:50]
                if not files:
                    rec['status']='INCONCLUSIVE_PROFILER_NO_TRACE'
                else:
                    rec['status']='DIAGNOSTIC_PROFILER_OVERHEAD'
            except Exception as e:
                rec['torch_profiler_error']=repr(e)
                rec['status']='INCONCLUSIVE_PROFILER_FAILED'
        for s in reversed(servers):
            with contextlib.suppress(Exception):s.stop()
        if servers:
            t_release=time.monotonic()
            while time.monotonic()-t_release<90:
                live=gpu_compute_pids()
                if live is not None and not (set(live)-{os.getpid()}):break
                time.sleep(3)
            rec['gpu_release_confirmed']=(live is not None and not (set(live)-{os.getpid()}))
            if not rec['gpu_release_confirmed']:
                rec['prior_status_before_cleanup_failure']=rec.get('status')
                rec['status']='FAIL_GPU_RELEASE'
                rec['reason']='GPU remains occupied or PID query unavailable; abort all subsequent arms' 
        rec['gpu_after']=gpu_inventory()
        rec['ended_utc']=datetime.now(timezone.utc).isoformat()

REPORTS={}
if MODE=='RUN':
    lab_start=time.monotonic()
    runnable=[a for a in SELECTED_ARMS if preflight[a]['artifact_ok']]
    if not runnable:
        for a in SELECTED_ARMS:
            REPORTS[a]={'arm':a,'status':'SKIPPED_REQUIRED_ARTIFACT','reason':preflight[a]['detail']}
        atomic_json(OUTPUT/'report.json',REPORTS)
        raise RuntimeError('NO_RUNNABLE_ARMS: attach model/artifacts before offline venv install')
    base=ensure_baseline_venv() if any(ARMS[x]['track']!='nccl_micro' for x in runnable) else None
    gpus=gpu_inventory()
    if len(gpus.get('data','').splitlines())<4:raise RuntimeError('4xGPU required; '+repr(gpus))
    TOK_CACHE={}
    try:
        for name in SELECTED_ARMS:
            if any(r.get('gpu_release_confirmed') is False for r in REPORTS.values()):
                print('HARD_ABORT_DIRTY_GPU: refusing additional arms',flush=True)
                break
            print('>>> ARM',name,flush=True)
            REPORTS[name]=single_arm_run(name,base,TOK_CACHE,lab_start)
            atomic_json(OUTPUT/'report.json',REPORTS)
            write_summary_csv(REPORTS)
            print(name,'=>',REPORTS[name]['status'],REPORTS[name].get('reason',''),flush=True)
    finally:
        atomic_json(OUTPUT/'report.json',REPORTS)
        write_summary_csv(REPORTS)
else:
    print('PLAN ONLY: edit MODE to RUN, select arms and attach offline wheel/model datasets.')

## 6. Post-run audit, honesty and acceptance criteria

The notebook's `PASS_BASIC_GATE` **is not** a production GO. Running MTP versus target-only needs a same-weight deterministic parity/first-high-margin-divergence suite with token IDs and logprobs; generated-thinking cache reuse and long GDN state continuity need long 8k→40k→90k→130k state tests. `nvfp4` and custom quant/host-AR need external numerical oracle and related patch provenance.

**Absolute blockers:** corrupted/empty output, missing output-token accounting, server restart masquerading as same arm, unverified wheel/patch, GPU memory OOM, unsupported SM89 kernel, rank communication errors, model load >10 minutes. **Positive speed criterion:** 40k/90k stable TPS **and** mixed-agent tokens/hour improve, while matched ARC task-level score and tool calls do not regress. Upgrade only after paired solve-rate checks.

In [ ]:
# Standalone report reader / plan-audit: NO GPU work. Run even in PLAN mode.
def audit_registry():
    keys=set(ARMS)
    required_tracks={'control','parallelism','custom_ar','graphs','lowm_fp8','nvfp4_kv','eagle4',
                     'profiling','local_argmax','ssm_state','nccl','spec_depth','hybrid_prefix','weight_quant'}
    got={a['track'] for a in ARMS.values()}
    assert required_tracks<=got, f"missing tracks {required_tracks-got}"
    assert all('requirement' in a for a in ARMS.values())
    assert all(a['name']==k for k,a in ARMS.items())
    assert ARMS['V0_FP8_BASE']['spec']==3
    assert ARMS['V11_TP1X4_REPLICAS']['replicas']==4
    assert ARMS['V2_FLASHINFER_FULL_PR60532']['requirement']=='patch_60532'
    assert ARMS['V12_HOST_STAGED_AR']['requirement']=='patch_host_ar'
    assert ARMS['V9_NVFP4_KV_TARGET_ONLY']['kv']=='nvfp4'
    assert ARMS['V10_EAGLE4']['requirement']=='eagle_spec'
    return {'arms_total':len(keys),'tracks':sorted(got),'selected':SELECTED_ARMS,
            'plan_artifact_status':preflight,'report_file':str(OUTPUT/'report.json')}
print(json.dumps(audit_registry(),indent=2,ensure_ascii=False))

### Exact source URLs (for Claude verification)

- https://github.com/vllm-project/vllm/tree/v0.32.0
- https://github.com/vllm-project/vllm/blob/v0.32.0/vllm/config/speculative.py
- https://github.com/vllm-project/vllm/blob/v0.32.0/vllm/config/cache.py
- https://github.com/vllm-project/vllm/pull/60532 — pre-SM90 multi-token verification FULL CUDA graphs.
- https://github.com/vllm-project/vllm/pull/61224 — Ada block-FP8 kernel (experimental source build).
- https://github.com/vllm-project/vllm/pull/46963 — NVFP4 KV handling; v0.32 runtime still needs SM89 verification.
- https://github.com/sgl-project/sglang/pull/39605 — host-staged collective engineering reference.
- https://github.com/NVIDIA/nccl-tests — NCCL microbenchmark.
- https://github.com/halt95/merlin-engine — TP1/low-M GDN experimental research.
- https://github.com/vllm-project/vllm/issues/63505 — hybrid PP correctness risk.
- https://github.com/vllm-project/vllm/issues/53670 — speculative cache block retention.
- https://github.com/vllm-project/vllm/issues/54919 — long prefill/decode starvation.
- https://github.com/vllm-project/vllm/issues/63593 — MTP rejection-state correctness.

**Important:** experimental arms with external dependencies are *registered, validated and independently selectable*, not prepackaged CUDA source patches. The notebook never falsely reports these as executed without the required external artifacts.
### Audit changes (2026-10-29)
- Corrected quadratic 90k/130k tokenizer work via logarithmic prompt repetition bisection and measured submitted chat-template lengths.
- Removed 8k-only `max_len` hardcode from TP1×4 and PP4 arms; long-context tests now attempt the selected context and report capacity failures truthfully.
- Added donor E21 vLLM 0.32 flags: `--enable-prompt-tokens-details`, `--trust-remote-code`, `--load-format safetensors`, `--dtype auto`, `--distributed-executor-backend mp`.
- Independent EAGLE / DFlash external spec paths and stricter local drafter-config preflight.
- NVFP4 SM89 requires a patch manifest with runtime evidence; passing the CLI parser alone never proves kernel support.
- Patched executable manifest must provide `source_file` and matching SHA256, not only a decorative hash.
- Fixed completion request-order preservation, wave timeout, short synthetic output classification, and 130k preflight.
- Added genuine target-only short text fingerprint with explicit `INCONCLUSIVE` when absent or differing. **This is not target logit parity.**
- Benchmark results still require Kaggle 4×L4 execution. No speedup claims are produced by this notebook in PLAN mode.

### Audit v3 — measurement-level changes
- `STANDARD` and `XL` benchmark each context twice: cold wave and warm identical-input wave. CPU-side prefill and decode are not conflated in labels.
- Streaming `/v1/chat/completions` with `stream_options.include_usage` measures TTFT plus a **first-token-to-last-completion window**. This window is not pure decode when initial-prefill finishes at different times; use the precise label.
- If server omits final streaming `completion_tokens`, arm FAILS; no token counts are fabricated. `Q27_STREAM_BENCHMARK=0` disables streaming but also removes TTFT metrics.
- Patched kernel activation is checked *after* benchmark execution, not only before first request.
- For 130k use `Q27_MODEL_LEN >= 130000 + output budget`; zero successful 130k tests means **partial**, never pass.

### Audit v4 — TRACE integrity and oracle isolation
- Per-run random `SESSION_ID` prevents accidental target-only oracle reuse after notebook/kernel restart.
- TRACE retains `tools`, `tool_choice`, `response_format` and relevant sampling metadata rather than flattening every request to messages. It **replays snapshots**, not a live agent with dynamic tool execution.
- NCCL-only microbenchmark does not spend minutes installing unrelated vLLM wheelhouse.

### Audit v5 — E21 donor parity and weight preflight
- E21 working v0.32 server flags restored: `--enable-auto-tool-choice`, `--tool-call-parser qwen3_coder`, `--reasoning-parser qwen3`, `--default-chat-template-kwargs {"preserve_thinking": true}` and `--disable-uvicorn-access-log`. CLI help is required to match.
- Checkpoint preflight now checks actual weight shard presence and every shard named in `.safetensors.index.json`, not just a decorative `config.json`.
- These checks are structural. The CUDA runtime, model outputs and performance still require actual Kaggle 4×L4 execution.

### Audit v6 — Merlin TP1 implementation gate
- `V11_TP1_SINGLE` and `V11_TP1X4_REPLICAS` now require `patch_merlin_tp1` executable/manifest **and** complete Merlin checkpoint shards. A Merlin quantized/host-embedding checkpoint alone does not enable its donor-specific loader in stock vLLM 0.32.
- PP4 remains baseline vLLM 0.32 high-risk research arm (known upstream Qwen hybrid PP issues); no false parity/throughput certification.

### Audit v7 — tool-calling response correctness
- OpenAI SSE tool-call deltas are reassembled by `index`, `name` and `arguments`; a valid tool-call-only answer is not misclassified as empty text.
- Malformed tool names or non-object JSON arguments fail the benchmark gate; request-level `tool_calls` are retained for the audit report.
- This is a static tool-response integrity check; full end-to-end agent tool execution still requires a live harness.

### Audit v8 — capacity instrumentation
- While requests run, sampler records `gpu_running_observed`, sampled KV usage fraction and per-card sampled GPU memory high-water. These are **sampled** observations, not true allocator peak; do not infer full 90k residency without active-count evidence.
- Baseline, warmed replay and trace/architecture arms each use the same sampling code.

- https://github.com/vllm-project/vllm/blob/v0.32.0/vllm/v1/metrics/loggers.py
- https://github.com/vllm-project/vllm/blob/v0.32.0/vllm/config/speculative.py
- https://github.com/vllm-project/vllm/blob/v0.32.0/vllm/config/cache.py
